<a href="https://colab.research.google.com/github/alexkuczera/PfDA_Assessment/blob/main/pfda_formative_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is my asssessment

In [25]:
corr_age_income = households["Average_Age_of_Residents"].corr(
    households["Median_Household_Income"]
)

fig = px.scatter(
    households,
    x="Average_Age_of_Residents",
    y="Median_Household_Income",
    trendline="ols",
    title="Correlation between Age and Income"
)
fig.show()

corr_age_income


np.float64(-0.013706286954778338)

In [26]:
hp = households.merge(properties, on="Address", how="inner")

corr_size_rooms = hp["Household_Size"].corr(hp["Number_of_Bedrooms"])

fig = px.scatter(
    hp,
    x="Household_Size",
    y="Number_of_Bedrooms",
    trendline="ols",
    title="Household Size vs Number of Bedrooms"
)
fig.show()

corr_size_rooms


np.float64(0.013271200771297486)

In [27]:
import statsmodels.api as sm

model_data = hp[[
    "Price",
    "Number_of_Bedrooms",
    "Number_of_Bathrooms",
    "Property_Size",
    "Year_Built",
    "Median_Household_Income"
]].dropna()

X = model_data.drop(columns=["Price"])
y = model_data["Price"]

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  Price   R-squared:                       0.917
Model:                            OLS   Adj. R-squared:                  0.917
Method:                 Least Squares   F-statistic:                 1.101e+04
Date:                Wed, 04 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:23:28   Log-Likelihood:                -58428.
No. Observations:                4999   AIC:                         1.169e+05
Df Residuals:                    4993   BIC:                         1.169e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===========================================================================================
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                    1.332e+04   2.24e+04      0.594      0.553   -3.07e+04    5.73e+04
Number_of_Bedrooms       7762.8051   1355.199      5.728      0.000    5106.019    1.04e+04
Number_of_Bathrooms      6128.1094   1207.754      5.074      0.000    3760.381    8495.838
Property_Size             133.2325      2.112     63.087      0.000     129.092     137.373
Year_Built                 -6.5143     11.419     -0.570      0.568     -28.900      15.871
Median_Household_Income    -0.0177      0.026     -0.685      0.494      -0.068       0.033
==============================================================================
Omnibus:                      471.292   Durbin-Watson:                   2.020
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              615.812
Skew:                          -0.858   Prob(JB):                    1.90e-134
Kurtosis:                       3.119   Cond. No.                     2.25e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.25e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
# ------------------------------------------------------------------------------------
# 0. Setup
# ------------------------------------------------------------------------------------
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from ipywidgets import interact, Dropdown, IntSlider, FloatSlider
from IPython.display import display

# Optional: if you have geo info later
# import geopandas as gpd

# ------------------------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------------------------
# ERROR EXPLANATION: The 'FileNotFoundError' indicates that the system could not find
# 'household_data.csv' or 'property_data.csv'. You need to ensure these files are
# uploaded to your Colab environment or provide the correct file paths.

# FIX: Assuming you intend to load these files directly into the 'households' and
# 'properties' DataFrames, here's a corrected version. The previous code had a redundant
# call to pd.read_csv and would have failed even if the files were found.
# Please update the file paths if your CSV files are located elsewhere, or upload
# 'household_data.csv' and 'property_data.csv' to your Colab environment.

# If you upload 'household_data.csv' and 'property_data.csv' directly to the Colab environment:
households = pd.read_csv('household_data.csv')
properties = pd.read_csv('property_data.csv', sep=None, engine="python")

# Example if your files were in '/content/sample_data/' and named differently:
# households = pd.read_csv('/content/sample_data/california_housing_train.csv')
# properties = pd.read_csv('/content/sample_data/california_housing_test.csv', sep=None, engine="python")

households.head(), properties.head()

(                Address Employment_Status Rent_vs_Own  Household_Size  \
 0  62 Academic Crescent           Retired         Own               2   
 1       55 Blossom Rise         Full-Time        Rent               1   
 2       104 Mill Street         Full-Time        Rent               6   
 3      93 George Square         Full-Time         Own               6   
 4      79 Faculty Court         Full-Time         Own               6   
 
    Average_Age_of_Residents  Median_Household_Income  
 0                        42              26086.25375  
 1                        35              57599.94313  
 2                        46              42927.94245  
 3                        47              47680.10089  
 4                        53              46356.53618  ,
    Property_ID     Neighborhood  Year_Built  Number_of_Bedrooms  \
 0            1       University        1934                   4   
 1            2  New Development        1925                   5   
 2           

In [ ]:
# ------------------------------------------------------------------------------------
# 2. Quick schema + basic checks
# ------------------------------------------------------------------------------------
print("Households info:")
display(households.info())
display(households.describe(numeric_only=True))

print("\nProperties info:")
display(properties.info())
display(properties.describe(numeric_only=True))

print("\nHousehold sample:")
display(households.sample(5, random_state=0))

print("\nProperty sample:")
display(properties.sample(5, random_state=0))


Households info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4999 entries, 0 to 4998
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Address                   4999 non-null   object 
 1   Employment_Status         4999 non-null   object 
 2   Rent_vs_Own               4999 non-null   object 
 3   Household_Size            4999 non-null   int64  
 4   Average_Age_of_Residents  4999 non-null   int64  
 5   Median_Household_Income   4999 non-null   float64
dtypes: float64(1), int64(2), object(3)
memory usage: 234.5+ KB


None

TypeError: NDFrame.describe() got an unexpected keyword argument 'numeric_only'

In [ ]:
# ------------------------------------------------------------------------------------
# 5. Join households and properties on Address
# ------------------------------------------------------------------------------------
hp = households.merge(
    properties,
    on="Address",
    how="inner",  # can change to 'left' if you want all households
    suffixes=("_HH", "_PROP")
)

print("Joined rows:", len(hp))
hp.head()


Joined rows: 4999


,Address,Employment_Status,Rent_vs_Own,Household_Size,Average_Age_of_Residents,Median_Household_Income,Property_ID,Neighborhood,Year_Built,Number_of_Bedrooms,Property_Size,Sold_Year,Number_of_Bathrooms,Property_Type,Price
0,62 Academic Crescent,Retired,Own,2,42,26086.25375,1,University,1934,4,1303,2013,4,Bungalow,242420
1,55 Blossom Rise,Full-Time,Rent,1,35,57599.94313,2,New Development,1925,5,1423,2011,5,Semi-detached,259990
2,104 Mill Street,Full-Time,Rent,6,46,42927.94245,3,Old Town,1967,3,1395,2016,3,Detached,184500
3,93 George Square,Full-Time,Own,6,47,47680.10089,4,City Centre,2005,5,2539,2022,5,Detached,455850
4,79 Faculty Court,Full-Time,Own,6,53,46356.53618,5,University,2011,5,2009,2011,5,Detached,356260


In [ ]:
# ------------------------------------------------------------------------------------
# 7. Descriptive analysis – tenure, income, employment
# ------------------------------------------------------------------------------------
# Tenure vs income
fig = px.box(
    households,
    x="Rent_vs_Own",
    y="Median_Household_Income",
    color="Rent_vs_Own",
    title="Household income distribution by tenure"
)
fig.show()

# Employment vs tenure
employment_tenure = (
    households
    .groupby(["Employment_Status", "Rent_vs_Own"])
    .size()
    .reset_index(name="Count")
)

fig = px.bar(
    employment_tenure,
    x="Employment_Status",
    y="Count",
    color="Rent_vs_Own",
    barmode="group",
    title="Employment status by tenure"
)
fig.show()

# Household size vs income
fig = px.scatter(
    households,
    x="Household_Size",
    y="Median_Household_Income",
    color="Rent_vs_Own",
    trendline="ols",
    title="Household size vs income by tenure"
)
fig.show()


In [ ]:
# ------------------------------------------------------------------------------------
# 8. Neighborhood-level indicators (policy lens)
# ------------------------------------------------------------------------------------

# Calculate derived columns before aggregation
hp['Persons_per_Bedroom'] = hp['Household_Size'] / hp['Number_of_Bedrooms']
hp['Overcrowded'] = (hp['Persons_per_Bedroom'] > 2).astype(int) # Example threshold
# Assuming 'Price' is property price and 'Median_Household_Income' is annual income
hp['Severely_Unaffordable'] = (hp['Price'] / hp['Median_Household_Income'] > 5).astype(int) # Example affordability ratio

neigh = (
    hp
    .groupby("Neighborhood")
    .agg(
        Avg_Income=("Median_Household_Income", "mean"),
        Median_Income=("Median_Household_Income", "median"),
        Avg_Price=("Price", "mean"),
        Median_Price=("Price", "median"),
        Overcrowding_Rate=("Overcrowded", "mean"),
        Severe_Unaffordability_Rate=("Severely_Unaffordable", "mean"),
        Rent_Share=("Rent_vs_Own", lambda s: (s == "Rent").mean()),
        Avg_Persons_per_Bedroom=("Persons_per_Bedroom", "mean"),
        HH_Count=("Property_ID", "count")
    )
    .reset_index()
)

neigh.sort_values("Severe_Unaffordability_Rate", ascending=False).head()

,Neighborhood,Avg_Income,Median_Income,Avg_Price,Median_Price,Overcrowding_Rate,Severe_Unaffordability_Rate,Rent_Share,Avg_Persons_per_Bedroom,HH_Count
3,University,29272.791590,27272.456450,252007.268698,248600.0,0.154017,0.774515,0.644321,1.367110,1805
0,City Centre,42821.017337,45406.561945,264061.380256,254400.0,0.163620,0.616088,0.393967,1.384796,1094
1,New Development,41406.681724,43025.713350,239057.818697,237210.0,0.137866,0.599622,0.389046,1.296868,1059
2,Old Town,43054.612316,45391.922180,197124.975985,200300.0,0.146974,0.438040,0.419789,1.352514,1041


In [ ]:
# Heatmap-style view of key indicators
fig = px.imshow(
    neigh.set_index("Neighborhood")[[
        "Avg_Income",
        "Avg_Price",
        "Overcrowding_Rate",
        "Severe_Unaffordability_Rate",
        "Rent_Share"
    ]].T,
    aspect="auto",
    color_continuous_scale="RdBu_r",
    title="Neighborhood-level housing and socio-economic indicators"
)
fig.update_layout(yaxis_title="Indicator", xaxis_title="Neighborhood")
fig.show()


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving household_data.csv to household_data.csv
Saving property_data.csv to property_data.csv


In [ ]:
# ------------------------------------------------------------------------------------
# 9. Interactive exploration widgets
# ------------------------------------------------------------------------------------
def explore_neighborhood(neighborhood, tenure, max_price_to_income):
    df = hp.copy()
    if neighborhood != "All":
        df = df[df["Neighborhood"] == neighborhood]
    if tenure != "All":
        df = df[df["Rent_vs_Own"] == tenure]
    df = df[df["Price_to_Income"] <= max_price_to_income]

    if df.empty:
        print("No records for this combination.")
        return

    fig = px.scatter(
        df,
        x="Median_Household_Income",
        y="Price",
        color="Employment_Status",
        size="Household_Size",
        hover_data=["Address", "Neighborhood", "Price_to_Income", "Persons_per_Bedroom"],
        title=f"Income vs Price – {neighborhood}, tenure={tenure}, PTI ≤ {max_price_to_income}"
    )
    fig.show()

    display(
        df[[
            "Address", "Neighborhood", "Rent_vs_Own", "Employment_Status",
            "Median_Household_Income", "Price", "Price_to_Income",
            "Persons_per_Bedroom", "Overcrowded"
        ]].head(10)
    )

neighborhood_options = ["All"] + sorted(hp["Neighborhood"].dropna().unique().tolist())
tenure_options = ["All"] + sorted(hp["Rent_vs_Own"].dropna().unique().tolist())

interact(
    explore_neighborhood,
    neighborhood=Dropdown(options=neighborhood_options, description="Neighborhood:"),
    tenure=Dropdown(options=tenure_options, description="Tenure:"),
    max_price_to_income=FloatSlider(value=10, min=1, max=20, step=0.5, description="Max PTI")
)


interactive(children=(Dropdown(description='Neighborhood:', options=('All', 'City Centre', 'New Development', …

<function __main__.explore_neighborhood(neighborhood, tenure, max_price_to_income)>

In [ ]:
# ------------------------------------------------------------------------------------
# 10. Policy-oriented summaries
# ------------------------------------------------------------------------------------
# 1) Neighborhoods with highest severe unaffordability
top_unaffordable = neigh.sort_values("Severe_Unaffordability_Rate", ascending=False).head(10)
print("Neighborhoods with highest severe unaffordability (Price_to_Income > 8):")
display(top_unaffordable)

# 2) Neighborhoods with highest overcrowding
top_overcrowded = neigh.sort_values("Overcrowding_Rate", ascending=False).head(10)
print("Neighborhoods with highest overcrowding (Persons_per_Bedroom > 2):")
display(top_overcrowded)

# 3) Neighborhoods with high rent share and low income (priority for affordable housing)
priority = neigh[
    (neigh["Rent_Share"] > neigh["Rent_Share"].median()) &
    (neigh["Median_Income"] < neigh["Median_Income"].median())
].sort_values("Rent_Share", ascending=False)

print("Priority neighborhoods for affordable rental interventions (high rent share, low income):")
display(priority.head(10))


Neighborhoods with highest severe unaffordability (Price_to_Income > 8):


,Neighborhood,Avg_Income,Median_Income,Avg_Price,Median_Price,Overcrowding_Rate,Severe_Unaffordability_Rate,Rent_Share,Avg_Persons_per_Bedroom,HH_Count
3,University,29272.791590,27272.456450,252007.268698,248600.0,0.154017,0.774515,0.644321,1.367110,1805
0,City Centre,42821.017337,45406.561945,264061.380256,254400.0,0.163620,0.616088,0.393967,1.384796,1094
1,New Development,41406.681724,43025.713350,239057.818697,237210.0,0.137866,0.599622,0.389046,1.296868,1059
2,Old Town,43054.612316,45391.922180,197124.975985,200300.0,0.146974,0.438040,0.419789,1.352514,1041


Neighborhoods with highest overcrowding (Persons_per_Bedroom > 2):


,Neighborhood,Avg_Income,Median_Income,Avg_Price,Median_Price,Overcrowding_Rate,Severe_Unaffordability_Rate,Rent_Share,Avg_Persons_per_Bedroom,HH_Count
0,City Centre,42821.017337,45406.561945,264061.380256,254400.0,0.163620,0.616088,0.393967,1.384796,1094
3,University,29272.791590,27272.456450,252007.268698,248600.0,0.154017,0.774515,0.644321,1.367110,1805
2,Old Town,43054.612316,45391.922180,197124.975985,200300.0,0.146974,0.438040,0.419789,1.352514,1041
1,New Development,41406.681724,43025.713350,239057.818697,237210.0,0.137866,0.599622,0.389046,1.296868,1059


Priority neighborhoods for affordable rental interventions (high rent share, low income):


,Neighborhood,Avg_Income,Median_Income,Avg_Price,Median_Price,Overcrowding_Rate,Severe_Unaffordability_Rate,Rent_Share,Avg_Persons_per_Bedroom,HH_Count
3,University,29272.79159,27272.45645,252007.268698,248600.0,0.154017,0.774515,0.644321,1.36711,1805


In [ ]:
# ------------------------------------------------------------------------------------
# 11. Narrative-style policy insights (printed text)
# ------------------------------------------------------------------------------------
def summarize_policy_findings():
    print("POLICY INSIGHTS SUMMARY\n")
    print("1. Affordability pressures")
    print("- Neighborhoods with high Price_to_Income ratios (e.g., those in 'top_unaffordable')")
    print("  indicate where households face the greatest difficulty purchasing housing.")
    print("- These areas are candidates for inclusionary zoning, shared ownership schemes,")
    print("  or targeted subsidies for first-time buyers.\n")

    print("2. Overcrowding hotspots")
    print("- Neighborhoods with high Overcrowding_Rate suggest a mismatch between household size")
    print("  and available bedroom capacity.")
    print("- Policy responses could include incentivising development of larger family units,")
    print("  revising occupancy standards, or prioritising these areas for new-build approvals.\n")

    print("3. Rental concentration and vulnerability")
    print("- Areas with high Rent_Share and low Median_Income are vulnerable to rent shocks and")
    print("  displacement.")
    print("- These neighborhoods are strong candidates for rent-stabilisation policies,")
    print("  expansion of social/affordable rental stock, and tenant support services.\n")

    print("4. Employment and tenure")
    print("- Cross-tabulating Employment_Status with Rent_vs_Own highlights groups at risk of housing")
    print("  insecurity (e.g., unemployed renters with low incomes).")
    print("- Interventions might include targeted employment programmes, housing benefit support,")
    print("  or pathways into intermediate tenure products.\n")

    print("5. Ageing and service planning")
    print("- Age_Band combined with tenure and income can reveal concentrations of older owner-occupiers")
    print("  with limited income but high housing equity.")
    print("- This can inform policies on equity release, age-friendly retrofits, and local health and")
    print("  social care provision.\n")

summarize_policy_findings()


POLICY INSIGHTS SUMMARY

1. Affordability pressures
- Neighborhoods with high Price_to_Income ratios (e.g., those in 'top_unaffordable')
  indicate where households face the greatest difficulty purchasing housing.
- These areas are candidates for inclusionary zoning, shared ownership schemes,
  or targeted subsidies for first-time buyers.

2. Overcrowding hotspots
- Neighborhoods with high Overcrowding_Rate suggest a mismatch between household size
  and available bedroom capacity.
- Policy responses could include incentivising development of larger family units,
  revising occupancy standards, or prioritising these areas for new-build approvals.

3. Rental concentration and vulnerability
- Areas with high Rent_Share and low Median_Income are vulnerable to rent shocks and
  displacement.
- These neighborhoods are strong candidates for rent-stabilisation policies,
  expansion of social/affordable rental stock, and tenant support services.

4. Employment and tenure
- Cross-tabulating Emp

In [ ]:
# ------------------------------------------------------------------------------------
# 0. Setup
# ------------------------------------------------------------------------------------
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from ipywidgets import interact, Dropdown, IntSlider, FloatSlider
from IPython.display import display

# Optional: if you have geo info later
# import geopandas as gpd

# ------------------------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------------------------
household_path = "household_data.csv"
property_path = "property_data.csv"

households = pd.read_csv(household_path)
properties = pd.read_csv(property_path, sep=None, engine="python")  # handles tab or comma

households.head(), properties.head()


(                Address Employment_Status Rent_vs_Own  Household_Size  \
 0  62 Academic Crescent           Retired         Own               2   
 1       55 Blossom Rise         Full-Time        Rent               1   
 2       104 Mill Street         Full-Time        Rent               6   
 3      93 George Square         Full-Time         Own               6   
 4      79 Faculty Court         Full-Time         Own               6   
 
    Average_Age_of_Residents  Median_Household_Income  
 0                        42              26086.25375  
 1                        35              57599.94313  
 2                        46              42927.94245  
 3                        47              47680.10089  
 4                        53              46356.53618  ,
    Property_ID     Neighborhood  Year_Built  Number_of_Bedrooms  \
 0            1       University        1934                   4   
 1            2  New Development        1925                   5   
 2           